# Privacy Toolkit — analysis notebook

Story: **hashing names is not anonymization.** We generate synthetic people, measure how unique they are on quasi-identifiers, show that unsalted hashes crack, replace identifiers with HMAC tokens, then actually anonymize (k-anonymity + l-diversity) and attack both releases with a fake voter list.

Prefer `python src/run_pipeline.py` for a full regenerate of `reports/`. This notebook walks through the same argument with the saved artifacts.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT / "src"))

from anonymize import anonymize
from paths import ANON_CSV, PSEUDO_CSV, RAW_CSV, REPORTS_DIR
from risk_metrics import summarize_risk
from utility import compare_utility, salary_by_age_band

raw = pd.read_csv(RAW_CSV, dtype={"zip_code": str}, keep_default_na=False)
print(raw.shape)
raw.head()

## 1. Baseline risk

Direct identifiers (name, email, phone) are obvious. **Quasi-identifiers** (age, ZIP, gender) are enough to single people out — that is the Sweeney result. Count equivalence classes on those three fields.

In [ ]:
baseline = summarize_risk(raw)
baseline

## 2. Unsalted hashing vs HMAC

`reports/pseudonymization_results.json` is written by the dictionary-attack script. Unsalted SHA-256 of emails is just a lookup table. HMAC with a secret key is not.

In [ ]:
pseudo_attack = json.loads((REPORTS_DIR / "pseudonymization_results.json").read_text())
pseudo_attack

## 3. Pseudonymized release

Names are gone. Age, ZIP, and gender are not. Linkage risk is unchanged.

In [ ]:
pseudo = pd.read_csv(PSEUDO_CSV, dtype={"zip_code": str}, keep_default_na=False)
print(pseudo.columns.tolist())
print("unique on age, ZIP, gender:", summarize_risk(pseudo)["pct_unique"], "%")
pseudo.head(3)

## 4. Anonymized release (k = 5, l = 2)

Age → bands, ZIP → 3 digits, drop classes that are too small or not diverse on `medical_condition`.

In [ ]:
anon = pd.read_csv(ANON_CSV, dtype={"zip3": str}, keep_default_na=False)
anon_stats = json.loads((REPORTS_DIR / "anonymization_results.json").read_text())
print(anon.shape)
anon_stats

## 5. Linkage attack

A public voter list (name + age + ZIP + gender) joined to each release. Unique matches on both sides are treated as re-identifications.

In [ ]:
json.loads((REPORTS_DIR / "linkage_results.json").read_text())

## 6. Utility: same questions, before and after

In [ ]:
util = compare_utility(raw, anon)
print("salary MAPE %:", util["salary_mape_pct"])
print("condition TV:", util["condition_total_variation"])
pd.DataFrame(
    {
        "raw": salary_by_age_band(raw),
        "anon": salary_by_age_band(anon),
    }
)

In [ ]:
pd.read_csv(REPORTS_DIR / "utility_vs_k.csv")

## 7. Charts

Open these files in `reports/`:

- `baseline_waffle.png` — share of records that are unique or below k = 5
- `baseline_group_sizes.png` — class-size histogram
- `risk_by_region.png` — uniqueness by ZIP prefix
- `linkage_attack.png` — voter-list join, pseudo vs anon
- `utility_vs_k.png` — accuracy lost as k grows
- `suppression_vs_k.png` — records dropped as k grows

Written conclusions: `reports/findings.md`.